In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.stats import zscore
from sklearn.preprocessing import QuantileTransformer

In [2]:
insurance_df=pd.read_csv(r"C:\Users\BAPS\Desktop\Python\Datasets\travel insurance.csv")
insurance_df
insurance_df.info()
# OBSERVATION -> There are 63326 records and 11 columns
# Columns > ['Agency', 'Agency Type', 'Distribution Channel', 'Product Name','Claim', 'Destination', 'Gender'] are object type rest are numeric['int64','float64']

insurance_df_o=insurance_df.copy()
insurance_df.select_dtypes('object').columns
((insurance_df.isnull().sum())/len(insurance_df))*100
total_missing = insurance_df.isnull().sum()
percentage_missing = total_missing * 100 / len(insurance_df)
missing_value_df = pd.DataFrame(data=[total_missing, percentage_missing], index=["Total", "%"]).T
missing_value_df

# OBSERVATION -> column ["Gender "] has 71% missing values
['Agency', 'Agency Type', 'Distribution Channel', 'Product Name',
       'Claim', 'Destination', 'Gender']
len(insurance_df["Destination"].unique())
len(insurance_df["Distribution Channel"].unique())
len(insurance_df["Product Name"].unique())
len(insurance_df["Agency"].unique())
len(insurance_df["Distribution Channel"].unique())
len(insurance_df["Agency Type"].unique())
len(insurance_df["Gender"].unique())
encoder=LabelEncoder()
insurance_df['Gender']=encoder.fit_transform(insurance_df['Gender'])
insurance_df['Agency']=encoder.fit_transform(insurance_df['Agency'])
insurance_df["Agency Type"]=encoder.fit_transform(insurance_df["Agency Type"])
insurance_df["Distribution Channel"]=encoder.fit_transform(insurance_df["Distribution Channel"])
insurance_df['Product Name']=encoder.fit_transform(insurance_df["Product Name"])
insurance_df["Destination"]=encoder.fit_transform(insurance_df["Destination"])
insurance_df['Claim']=encoder.fit_transform(insurance_df['Claim'])
insurance_df.drop(["Gender"],axis=1,inplace=True)
# Since gender is binary categorical data and has 71% missing values,it is not feasible to handle missing values,therefore droping this column
insurance_df.drop(columns=["Destination"],inplace=True)
# Droping this column because, there are 149 unique values

insurance_df.describe().T

##### z-score
print('Before outlier removal : ',insurance_df.shape)
z = np.abs(zscore(insurance_df, axis = 1))
threshold = 3
dataset = insurance_df[(z < threshold).all(axis=1)]
print('After Z-Score approach : ',dataset.shape)
Q1 = insurance_df.quantile(0.25)  # Q1
Q3 = insurance_df.quantile(0.75)  # Q3
IQR = Q3-Q1
dataset = insurance_df[~((insurance_df<(Q1-1.5*IQR)) | (insurance_df>(Q3+1.5*IQR)))]
print('After IQR approach : ',dataset.shape)
cond1=insurance_df['Duration']<1000
in_df=insurance_df.where(cond1)
in_df.dropna()
cond2=in_df["Age"]<100
in_df=in_df.where(cond2)
in_df.dropna()
cond3=in_df["Commision (in value)"]<200
in_df=in_df.where(cond3)
in_df.dropna()
cond4=in_df["Net Sales"]<600
in_df=in_df.where(cond4)
in_df.dropna()
cond5=in_df["Age"]>0
in_df=in_df.where(cond5)
in_df.dropna()
cond6=in_df["Net Sales"]>-300
in_df=in_df.where(cond6)
in_df.dropna()
in_df.skew()
in_df.kurt()
quantile_transformer = QuantileTransformer(output_distribution='normal')
in_df["Commision (in value)"]= quantile_transformer.fit_transform(in_df["Commision (in value)"].values.reshape(-1, 1)).flatten()
quantile_transformer = QuantileTransformer(output_distribution='normal')
in_df["Net Sales"]= quantile_transformer.fit_transform(in_df["Net Sales"].values.reshape(-1, 1)).flatten()
quantile_transformer = QuantileTransformer(output_distribution='normal')
in_df["Age"]= quantile_transformer.fit_transform(in_df["Age"].values.reshape(-1, 1)).flatten()
quantile_transformer = QuantileTransformer(output_distribution='normal')
in_df["Duration"]= quantile_transformer.fit_transform(in_df["Duration"].values.reshape(-1, 1)).flatten()
in_df.dropna(inplace=True)
from sklearn.model_selection import train_test_split
X=in_df.drop(["Agency Type"],axis=1)
Y=in_df["Agency Type"]
x_train,x_test,y_train,y_test=train_test_split(X,Y,train_size=0.80)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)
#### Scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63326 entries, 0 to 63325
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Agency                63326 non-null  object 
 1   Agency Type           63326 non-null  object 
 2   Distribution Channel  63326 non-null  object 
 3   Product Name          63326 non-null  object 
 4   Claim                 63326 non-null  object 
 5   Duration              63326 non-null  int64  
 6   Destination           63326 non-null  object 
 7   Net Sales             63326 non-null  float64
 8   Commision (in value)  63326 non-null  float64
 9   Gender                18219 non-null  object 
 10  Age                   63326 non-null  int64  
dtypes: float64(2), int64(2), object(7)
memory usage: 5.3+ MB
Before outlier removal :  (63326, 9)
After Z-Score approach :  (63326, 9)
After IQR approach :  (63326, 9)
(49810, 8)
(12453, 8)
(49810,)
(12453,)


In [3]:
## Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate
lr = make_pipeline(
    LogisticRegression(max_iter=2000))
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
specificity = cm[1,1] / (cm[1,0] + cm[1,1])
print("Specificity -> ",specificity)

Accuracy: 0.7945073476270779

Confusion Matrix:
 [[2398  859]
 [1700 7496]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.59      0.74      0.65      3257
         1.0       0.90      0.82      0.85      9196

    accuracy                           0.79     12453
   macro avg       0.74      0.78      0.75     12453
weighted avg       0.82      0.79      0.80     12453

Specificity ->  0.815137016093954


Agency Type
1.0    45719
0.0    16544
Name: count, dtype: int64

### Single layer Perceptron

In [4]:
import tensorflow as tf


c:\Users\BAPS\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\BAPS\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\BAPS\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/

In [19]:
model=tf.keras.Sequential([tf.keras.layers.Dense(1,activation="sigmoid",input_shape=(8,))])

c:\Users\BAPS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [ ]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)
hist=model.fit(x_train,y_train,epochs=300,batch_size=20,validation_split=0.3,verbose=0)
print(hist)
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")
print("Loss -> ",loss)

(49810, 8) (12453, 8) (49810,) (12453,)
Test Accuracy: 0.7915
Loss ->  0.31448015570640564


### MultiLayer Perceptron

In [9]:
from tensorflow.keras.layers import Dense,LeakyReLU 

In [16]:
model=tf.keras.models.Sequential([Dense(64,LeakyReLU(alpha=0.03)),
                                  Dense(32,activation="sigmoid"),
                                  Dense(1,activation="sigmoid")])

model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

c:\Users\BAPS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [17]:
mod=model.fit(X_train_scaled,y_train,epochs=20,batch_size=2000,validation_split=0.2)
print(mod)

Epoch 1/20


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.3897 - loss: 0.7391 - val_accuracy: 0.7503 - val_loss: 0.6496
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7405 - loss: 0.5954 - val_accuracy: 0.7338 - val_loss: 0.5460
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7707 - loss: 0.5035 - val_accuracy: 0.7952 - val_loss: 0.4649
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8137 - loss: 0.4291 - val_accuracy: 0.8146 - val_loss: 0.3960
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8409 - loss: 0.3664 - val_accuracy: 0.8738 - val_loss: 0.3386
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8965 - loss: 0.3142 - val_accuracy: 0.9302 - val_loss: 0.2907
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9433 - loss: 0.2698 - val_accuracy: 0.9563 - val_loss: 0.2494
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9626 - loss: 0.2308 - val_accuracy: 0.9658 - val_loss: 0.2130
Ep

In [18]:
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
accuracy*=100
print(f"Test Accuracy: {accuracy:.5f}")
print("Loss -> ",loss)

Test Accuracy: 97.36609
Loss ->  0.07974252849817276
